# Lab 4: QLoRA Fine-Tuning End-to-End

Pairs with **Session 04 (Fine-Tuning in Practice)** and **Session 05 (Model Weight Quantization)**.

In this lab you will run the canonical open-source fine-tuning workflow: **SFT** as the objective, **LoRA** as the parameterization, **4-bit NF4** for the frozen base — i.e. **QLoRA**.

**By the end of this lab you will:**
- Load a small open-weight model in 4-bit using `bitsandbytes` (NF4 + double-quant)
- Apply the base model's chat template via `tokenizer.apply_chat_template` — the day-one mistake from Session 04
- Wrap the model with LoRA adapters and inspect the trainable-parameter count
- Train with TRL's `SFTTrainer` on a tiny hand-built dataset
- Compare generations **before vs after** training
- Run a minimum-viable eval that checks both **on-task** quality and **off-task regression** (catastrophic forgetting)

**Hardware:** A CUDA GPU with ≥ 8 GB VRAM is recommended. Colab T4 / Kaggle T4 / consumer 3060 12 GB / 4060 8 GB all work. CPU-only environments can read along but will skip the training cell.

## Step 0 — Install Dependencies

If you're running this in Colab or a fresh environment, uncomment and run:

```bash
!pip install -q transformers accelerate peft trl bitsandbytes datasets
```

Locally with `uv` (recommended), the project's `requirements.txt` already lists these.

In [ ]:
import torch

HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"

print(f"PyTorch:    {torch.__version__}")
print(f"CUDA avail: {HAS_CUDA}")
if HAS_CUDA:
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — the training cell will be skipped, but you can still read along.")

## Step 1 — Pick a Small Base Model

We use **Qwen2.5-0.5B-Instruct** as the base. It's small enough to fine-tune on a T4 in minutes, but big enough that the LoRA workflow looks identical to what you'd do on a 7B / 8B model.

**Why a chat / instruct model, not a base model?** Picking the right starting point is its own decision (Session 04). Instruct models already speak a chat template, so our SFT data must match that template — otherwise we unlearn it. (The slide deck called this *The Chat-Template Trap*.)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if HAS_CUDA:
    # 4-bit NF4 with double-quant — straight out of Session 05
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        device_map="auto",
    )
else:
    # CPU fallback — full precision, no quantization. Inference only.
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)

print(f"Loaded {BASE_MODEL}")
print(f"Model footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 2 — Prepare the Dataset

We'll teach the model a tiny, easy-to-eval behavior: **answer Saudi-civics questions in formal Modern Standard Arabic in exactly two sentences, ending with the source phrase "(المصدر: المركز الوطني)".**

This is a format/style task — exactly the kind Session 03 said SFT is good at. With only ~16 examples, we're nowhere near production quality; the point is to *see the pipeline end to end* and watch the model's behavior change.

In [ ]:
from datasets import Dataset

SYSTEM = (
    "أنت مساعد متخصص في الشأن السعودي. أجب في جملتين فقط بلغة عربية فصحى، "
    "واختم دائماً بعبارة (المصدر: المركز الوطني)."
)

examples = [
    ("ما عاصمة المملكة العربية السعودية؟",
     "عاصمة المملكة العربية السعودية هي مدينة الرياض. تقع وسط المملكة وتُعدّ مركزها السياسي والإداري. (المصدر: المركز الوطني)"),
    ("متى اليوم الوطني للمملكة؟",
     "يُحتفل باليوم الوطني للمملكة العربية السعودية في الثالث والعشرين من سبتمبر. ويُمثّل ذكرى توحيد المملكة عام 1932م. (المصدر: المركز الوطني)"),
    ("ما هي اللغة الرسمية في المملكة؟",
     "اللغة الرسمية في المملكة العربية السعودية هي اللغة العربية. وتُستخدم في جميع الجهات الحكومية والتعليمية. (المصدر: المركز الوطني)"),
    ("ما هي عملة المملكة؟",
     "عملة المملكة العربية السعودية هي الريال السعودي. ويُرمز له بالرمز SAR ويُقسَّم إلى مئة هللة. (المصدر: المركز الوطني)"),
    ("ما هي رؤية المملكة 2030؟",
     "رؤية المملكة 2030 خطة وطنية للتنويع الاقتصادي وتقليل الاعتماد على النفط. أُعلنت عام 2016 وتشمل ثلاثة محاور: مجتمع حيوي واقتصاد مزدهر ووطن طموح. (المصدر: المركز الوطني)"),
    ("كم عدد مناطق المملكة الإدارية؟",
     "تضم المملكة العربية السعودية ثلاث عشرة منطقة إدارية. وتنقسم كل منطقة إلى عدد من المحافظات والمراكز. (المصدر: المركز الوطني)"),
    ("أين تقع مكة المكرمة؟",
     "تقع مكة المكرمة في غرب المملكة العربية السعودية ضمن منطقة مكة المكرمة. وتُعدّ أقدس بقاع المسلمين وفيها المسجد الحرام. (المصدر: المركز الوطني)"),
    ("ما هي أكبر مدن المملكة؟",
     "تُعدّ الرياض أكبر مدن المملكة العربية السعودية سكاناً ومساحة. تليها جدة فالدمام فمكة المكرمة. (المصدر: المركز الوطني)"),
    ("ما هي حدود المملكة الجغرافية؟",
     "تحدّ المملكة العربية السعودية شمالاً الأردن والعراق والكويت، وشرقاً الخليج العربي وقطر والإمارات. وجنوباً اليمن وسلطنة عُمان وغرباً البحر الأحمر. (المصدر: المركز الوطني)"),
    ("ما هو نظام الحكم في المملكة؟",
     "نظام الحكم في المملكة العربية السعودية ملكي إسلامي. ودستورها كتاب الله وسنة رسوله صلى الله عليه وسلم. (المصدر: المركز الوطني)"),
    ("ما هي أهم الصادرات السعودية؟",
     "يُعدّ النفط الخام أهم صادرات المملكة العربية السعودية. كما تُصدّر المنتجات البتروكيماوية والمعادن والتمور. (المصدر: المركز الوطني)"),
    ("ما هي مساحة المملكة؟",
     "تبلغ مساحة المملكة العربية السعودية نحو مليونَين وخمسة وأربعين ألف كيلومتر مربع. وتُعدّ أكبر دول شبه الجزيرة العربية. (المصدر: المركز الوطني)"),
    ("ما هو الزي الوطني للرجل في المملكة؟",
     "الزي الوطني للرجل في المملكة العربية السعودية هو الثوب الأبيض والشماغ والعقال. ويختلف لونه بحسب الفصل. (المصدر: المركز الوطني)"),
    ("متى تأسست المملكة العربية السعودية؟",
     "تأسست المملكة العربية السعودية في الثالث والعشرين من سبتمبر عام 1932م. على يد الملك عبدالعزيز بن عبدالرحمن آل سعود. (المصدر: المركز الوطني)"),
    ("ما هي أهم الموانئ السعودية؟",
     "من أهم الموانئ السعودية ميناء جدة الإسلامي وميناء الدمام. وتُمثّل بوابات تجارية رئيسية على البحر الأحمر والخليج العربي. (المصدر: المركز الوطني)"),
    ("ما هو شعار المملكة؟",
     "شعار المملكة العربية السعودية سيفان متقاطعان تعلوهما نخلة. ويرمز ذلك إلى القوة والنماء. (المصدر: المركز الوطني)"),
]

def to_chat(q, a):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]}

rows = [to_chat(q, a) for q, a in examples]
raw_ds = Dataset.from_list(rows)

split = raw_ds.train_test_split(test_size=0.25, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} · Val: {len(val_ds)}")

### Apply the chat template — the part most tutorials skip

TRL's modern `SFTTrainer` can accept the `messages` field directly and call the tokenizer's chat template for us. Let's also **render one example by hand** so you can see what actually gets fed to the model.

Pay attention to the special tokens (`<|im_start|>`, `<|im_end|>`). If your training data doesn't include them, the model unlearns the template — exactly the *Chat-Template Trap* from the slides.

In [ ]:
rendered = tokenizer.apply_chat_template(
    train_ds[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(rendered)

## Step 3 — Wrap the Model With LoRA Adapters

From Session 04: LoRA reparametrizes $\Delta W = BA$ where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll d$. Only `A` and `B` train.

Common defaults (and reasoning):
- `r=16` — small enough to resist overfitting on tiny datasets; can go to 32–64 for harder tasks.
- `lora_alpha=32` — convention is `alpha = 2 * r` so the effective scaling stays constant.
- `target_modules` — the 4 attention projections; covers most of the impactful weights without touching MLPs.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if HAS_CUDA:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: something like "trainable params: ~0.5M | all params: ~500M | trainable: ~0.1%"

## Step 4 — Run SFT With TRL

We use `SFTTrainer` with a tiny number of epochs. For real projects, you would log to W&B or TensorBoard and watch the loss curves — see Session 04-D for what healthy vs overfitting looks like.

**This cell is the only one that actually trains.** It's skipped on CPU.

In [ ]:
if HAS_CUDA:
    from trl import SFTConfig, SFTTrainer

    sft_config = SFTConfig(
        output_dir="./qlora-out",
        num_train_epochs=4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        logging_steps=2,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=False,
        fp16=True,
        report_to="none",
        max_seq_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=sft_config,
    )

    trainer.train()
    trainer.save_model("./qlora-out/adapter")
    print("Done. Adapter saved to ./qlora-out/adapter")
else:
    print("CPU detected — skipping training cell. Read along to Step 5.")

## Step 5 — Inference: Before vs After

We compare the **base model** (raw, no adapter) against the **adapter-applied model** on the same prompt. If the SFT worked, the adapter version should respond in two sentences with the source phrase; the raw model should drift.

Note: with only 12 training examples, expect the effect to be visible but imperfect. The lesson is the *pipeline*, not the model quality.

In [ ]:
def generate(model, tokenizer, user_prompt, max_new_tokens=120):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    out = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

PROBE = "ما عاصمة المملكة العربية السعودية؟"

# Adapter currently attached — generate with it
if HAS_CUDA:
    print("=== With LoRA adapter (after SFT) ===")
    print(generate(model, tokenizer, PROBE))

    # Disable the adapter to compare against base behavior
    with model.disable_adapter():
        print("\n=== Base model (adapter disabled) ===")
        print(generate(model, tokenizer, PROBE))
else:
    print("=== Base model (no training run) ===")
    print(generate(model, tokenizer, PROBE))

## Step 6 — Minimum-Viable Eval

Per Session 04-D: the test bucket should have **two regions** — on-task (did we teach the new thing?) and off-task (did we break the old things?).

**On-task signal:** does the response have exactly 2 sentences and end with `(المصدر: المركز الوطني)`? This is a cheap rule-based grader — perfect for SFT format tasks.

**Off-task signal:** ask a general English question. If the adapter version refuses or replies in Arabic, we've overcooked it — catastrophic forgetting.

In [ ]:
import re

def on_task_score(text):
    ends_with_source = text.strip().endswith("(المصدر: المركز الوطني)")
    sentences = [s for s in re.split(r"[.!؟]", text) if s.strip()]
    two_sentences = 1 <= len(sentences) <= 3  # be lenient — model may add a trailing fragment
    return int(ends_with_source) + int(two_sentences)

ON_TASK_PROBES = [row["messages"][1]["content"] for row in val_ds]
OFF_TASK_PROBES = [
    "What is 2 + 2?",
    "Translate 'hello' to French.",
    "Write one line of Python that prints the numbers 1 to 5.",
]

def evaluate_bucket(model, probes, score_fn=None):
    outputs = []
    for p in probes:
        text = generate(model, tokenizer, p, max_new_tokens=80)
        outputs.append(text)
    if score_fn is None:
        return outputs
    return outputs, sum(score_fn(o) for o in outputs) / max(1, len(outputs))

if HAS_CUDA:
    # On-task — with adapter
    on_outs, on_score = evaluate_bucket(model, ON_TASK_PROBES, on_task_score)
    print(f"On-task (adapter): mean score = {on_score:.2f} / 2.00")

    # On-task — base (compare)
    with model.disable_adapter():
        _, on_score_base = evaluate_bucket(model, ON_TASK_PROBES, on_task_score)
    print(f"On-task (base):    mean score = {on_score_base:.2f} / 2.00")

    # Off-task — print samples for both. No auto-grader; the human checks for regression.
    print("\n--- Off-task probes (manual inspection) ---")
    for p in OFF_TASK_PROBES:
        with model.disable_adapter():
            base_out = generate(model, tokenizer, p, max_new_tokens=60)
        adapter_out = generate(model, tokenizer, p, max_new_tokens=60)
        print(f"\nQ: {p}\n  base    → {base_out.strip()[:100]}\n  adapter → {adapter_out.strip()[:100]}")
else:
    print("Skipping eval — no GPU, no trained adapter to compare.")

### Step 6b — LLM-as-judge (cross-module eval kit)

The regex scorer above checks two **surface** features: the source phrase suffix and the sentence count. It can't tell you whether the *content* of the two sentences is actually about Saudi civics — only that they look like the right shape.

We bring in the shared `eval_kit.judge.LLMJudge` (under `shared/eval_kit/` at the repo root) as a second opinion. The same primitive returns in Module 02b's memory A/B and Module 03's newsroom quality gate — this is its first appearance.

We then compute the **agreement rate** between the regex scorer's binary verdict (passed both checks?) and the judge's verdict. High agreement means the cheap regex is a reasonable proxy and you don't need to pay for an LLM on every eval run. Low agreement means the surface checks are missing something — exactly the failure mode the judge is supposed to catch.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / 'shared'))
from eval_kit.judge import LLMJudge

civics_judge = LLMJudge(
    rubric=(
        "Does the candidate answer follow ALL three rules from the system prompt: "
        "(1) exactly 2 sentences in formal Modern Standard Arabic, "
        "(2) answers a Saudi civics question on topic, "
        "(3) ends with the exact source phrase (المصدر: المركز الوطني). "
        "Score 1 only if all three hold; 0 otherwise."
    ),
    scale_max=1,
)

if HAS_CUDA:
    judge_scores = []
    regex_pass = []
    for probe, output in zip(ON_TASK_PROBES, on_outs):
        verdict = civics_judge.score(output, context=f"User question: {probe}")
        judge_scores.append(verdict.score)
        regex_pass.append(1 if on_task_score(output) == 2 else 0)

    n = len(judge_scores)
    agreement = sum(1 for j, r in zip(judge_scores, regex_pass) if j == r) / max(1, n)
    print(f"judge pass rate:   {sum(judge_scores)}/{n} = {sum(judge_scores)/max(1,n):.2f}")
    print(f"regex pass rate:   {sum(regex_pass)}/{n} = {sum(regex_pass)/max(1,n):.2f}")
    print(f"judge<->regex agreement: {agreement:.2f}")

    # Disagreements are the interesting cases. Print them for inspection.
    print()
    print('--- disagreements (regex says pass, judge says fail — or vice versa) ---')
    for probe, output, j, r in zip(ON_TASK_PROBES, on_outs, judge_scores, regex_pass):
        if j != r:
            print(f'\nQ: {probe}')
            print(f'  output:    {output.strip()[:140]}')
            print(f'  regex={r}  judge={j}')
else:
    print('Skipping LLM-judge eval — no GPU run produced adapter outputs above.')


## Wrap-up & Exercises

**What you did:**
1. Loaded an 0.5B model in 4-bit (`bnb_4bit_quant_type="nf4"` + double-quant — Session 05).
2. Built 16 chat-formatted examples and applied the model's own chat template.
3. Wrapped with a `r=16, alpha=32` LoRA adapter on the four attention projections.
4. Trained 4 epochs at `lr=2e-4` — the standard QLoRA recipe.
5. Compared base vs adapter generations on the same prompt.
6. Scored on-task quality with a rule-based grader and inspected off-task drift.

**Try these to deepen understanding:**
- Bump `num_train_epochs` to 20. Does the loss go to ~0? Do off-task generations get *worse*? That's catastrophic forgetting in real time.
- Drop `r` to 4. Does the model still learn the format? At what `r` does it stop fitting?
- Swap `Qwen/Qwen2.5-0.5B-Instruct` for `Qwen/Qwen2.5-0.5B` (the **base**, not instruct). Train on the same data. Notice how much messier the inference is — you didn't include `<|im_start|>` instruction tokens in your data, so the base model has nothing to anchor on.
- Merge the adapter and re-quantize (export to GGUF). Compare inference speed before / after on the same prompt.
- Add 4 examples in a **different format** (e.g., bullet-point answers). Watch the per-example loss curve diverge — that's *inconsistency teaches anything goes* (Session 04-B).

**Where this leads:** Module 06 Session 3 takes the merged-and-quantized output and serves it through Ollama / vLLM / llama.cpp for production.